In [1]:
from transformers import AutoTokenizer, AutoModel
# from bertviz.transformers_view_neurons import BertModel
# from bertviz.neuron_view import show
import os

In [2]:
# base = "C:/Users/98922/.cache/huggingface/hub/models--dbmdz--bert-large-cased-finetuned-conll03-english"
# snap = os.path.join(base, "snapshots", os.listdir(os.path.join(base, "snapshots"))[0])

In [2]:
# from bertviz.transformers_neuron_view import BertModel
# from bertviz.neuron_view import show
model_ckpt = "bert-base-uncased"
# model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModel.from_pretrained(model_ckpt)
# model = BertModel.from_pretrained(model_ckpt)
text = "time flies like an arrow"
# show(model, "bert", tokenizer, text, display_mode="light", layer=0, head=8)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [4]:
print(model)

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [7]:
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
inputs.input_ids

tensor([[ 2051, 10029,  2066,  2019,  8612]])

In [8]:
from transformers import AutoConfig
from torch import nn

In [9]:
config = AutoConfig.from_pretrained(model_ckpt)
token_emb = nn.Embedding(config.vocab_size, config.dim)
token_emb

Embedding(30522, 768)

In [10]:
input_embds = token_emb(inputs.input_ids)
input_embds.size()

torch.Size([1, 5, 768])

In [11]:
import torch
from math import sqrt
query = key = value = input_embds
dim_k = key.size(-1)
scores = torch.bmm(query, key.transpose(1, 2)) / sqrt(dim_k)
scores.size()

torch.Size([1, 5, 5])

In [12]:
import torch.nn.functional as F

weights = F.softmax(scores, dim=-1)
weights.sum(dim=-1)

tensor([[1., 1., 1., 1., 1.]], grad_fn=<SumBackward1>)

In [13]:
attn_outputs = torch.bmm(weights, value)
attn_outputs.shape

torch.Size([1, 5, 768])

In [14]:
def scaled_dot_product_attention(query, key, value):
    dim_k = key.size(-1)
    scores = torch.bmm(query, key.transpose(1, 2)) / sqrt(dim_k)
    weights = F.softmax(scores, dim=-1)
    attn = torch.bmm(weights, value)
    return attn

In [15]:
class AttentionHead(nn.Module):
    def __init__(self, emb_dim, head_dim):
        super().__init__()
        self.q = nn.Linear(emb_dim, head_dim)
        self.k = nn.Linear(emb_dim, head_dim)
        self.v = nn.Linear(emb_dim, head_dim)

    def forward(self, hidden_state):
        attn_outputs = scaled_dot_product_attention(
            self.q(hidden_state), self.k(hidden_state), self.v(hidden_state)
        )
        return attn_outputs

In [23]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        emb_dim = config.dim
        num_heads = config.n_heads
        head_dim = emb_dim // num_heads
        self.heads = nn.ModuleList(
            [AttentionHead(emb_dim, head_dim) for _ in range(num_heads)]
        )
        self.output_linear = nn.Linear(head_dim*num_heads, emb_dim)

    def forward(self, hidden_state):
        x = torch.cat([h(hidden_state) for h in self.heads], dim=-1)
        x = self.output_linear(x)
        return x

In [17]:
multihead_attn = MultiHeadAttention(config)
attn_outputs = multihead_attn(input_embds)
attn_outputs.size()

torch.Size([1, 5, 768])

In [18]:
# from bertviz import head_view
from transformers import AutoModel

model = AutoModel.from_pretrained(model_ckpt, output_attentions=True)

In [19]:
sentence_a = "time flies like an arrow"
sentence_b = "fruit flies like a banana"

In [20]:
viz_inputs = tokenizer(sentence_a, sentence_b, return_tensors="pt")
attention = model(**viz_inputs).attentions

In [22]:
# sentence_b_start = (viz_inputs.token_type_ids == 0).sum(dim=1)
tokens = tokenizer.convert_ids_to_tokens(viz_inputs.input_ids[0])
# head_view(attention, tokens, sentence_b_start, heads=[8])

In [63]:
# head_view(attention, tokens, sentence_b_start, heads[8])